# 🛡️ Day 05a: Middleware & CORS — The Request Interceptors

---

## 🎯 What You'll Master Today
- What middleware is and the request/response lifecycle
- Custom middleware (logging, timing, auth)
- CORS — Cross-Origin Resource Sharing
- Exception handlers (global error handling)
- Startup & shutdown events (lifespan)

---

## 🎭 The Security Guard Analogy

Middleware is like a security checkpoint at a building entrance. Every person (request) passes through on the way in, and every person passes through again on the way out. The guard can check IDs, log entry/exit, or even deny entry.

```
╔═══════════════════════════════════════════════════════════════════╗
║  REQUEST → MIDDLEWARE STACK → ENDPOINT                           ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Client Request                                                   ║
║    │                                                              ║
║    ▼                                                              ║
║  ┌─────────────┐                                                  ║
║  │ Middleware 1 │ ← runs FIRST on request, LAST on response      ║
║  └──────┬──────┘                                                  ║
║    ▼                                                              ║
║  ┌─────────────┐                                                  ║
║  │ Middleware 2 │ ← runs SECOND on request                       ║
║  └──────┬──────┘                                                  ║
║    ▼                                                              ║
║  ┌─────────────┐                                                  ║
║  │  Endpoint   │ ← your route handler                            ║
║  └──────┬──────┘                                                  ║
║    ▼                                                              ║
║  Response flows back UP through middleware stack                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Custom Middleware — Request Timing
# ============================================================
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
import time

app = FastAPI()

@app.middleware("http")
async def add_process_time(request: Request, call_next):
    """
    Timing middleware — adds X-Process-Time header.
    
    Pattern:
    1. Do something BEFORE (request phase)
    2. response = await call_next(request)  ← passes to next middleware/endpoint
    3. Do something AFTER (response phase)
    4. Return response
    """
    start = time.perf_counter()
    response = await call_next(request)
    duration = time.perf_counter() - start
    response.headers["X-Process-Time"] = f"{duration:.4f}s"
    return response

@app.get("/fast")
def fast_endpoint():
    return {"speed": "fast"}

@app.get("/slow")
async def slow_endpoint():
    import asyncio
    await asyncio.sleep(0.1)
    return {"speed": "slow"}

client = TestClient(app)
r = client.get("/fast")
print(f"Fast: {r.headers['X-Process-Time']}")

r = client.get("/slow")
print(f"Slow: {r.headers['X-Process-Time']}")

In [ ]:
# ============================================================
# Request Logging Middleware
# ============================================================
import uuid

app2 = FastAPI()

@app2.middleware("http")
async def log_requests(request: Request, call_next):
    """Log every request with a unique request ID."""
    request_id = str(uuid.uuid4())[:8]
    
    # Log request
    print(f"[{request_id}] → {request.method} {request.url.path}")
    
    start = time.perf_counter()
    response = await call_next(request)
    duration = time.perf_counter() - start
    
    # Log response
    print(f"[{request_id}] ← {response.status_code} ({duration:.3f}s)")
    
    # Add request ID to response for tracing
    response.headers["X-Request-ID"] = request_id
    return response

@app2.get("/hello")
def hello():
    return {"msg": "Hello!"}

@app2.get("/error")
def error_endpoint():
    raise HTTPException(500, "Something broke")

client2 = TestClient(app2, raise_server_exceptions=False)
print("--- Successful request: ---")
r = client2.get("/hello")
print(f"Request ID: {r.headers['X-Request-ID']}")
print("\n--- Error request: ---")
r = client2.get("/error")
print(f"Status: {r.status_code}")

In [ ]:
# ============================================================
# CORS — The #1 Frontend-Backend Issue
# ============================================================
from fastapi.middleware.cors import CORSMiddleware

app3 = FastAPI()

# CORS tells the browser: "it's OK for these origins to call me"
app3.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",      # React dev server
        "http://localhost:5173",      # Vite dev server
        "https://myapp.com",         # Production frontend
    ],
    allow_credentials=True,           # Allow cookies
    allow_methods=["*"],              # GET, POST, PUT, DELETE, etc.
    allow_headers=["*"],              # Authorization, Content-Type, etc.
    expose_headers=["X-Request-ID"], # Custom headers visible to frontend
)

@app3.get("/api/data")
def get_data():
    return {"data": "this works cross-origin"}

print("CORS Configuration:")
print("✅ allow_origins → list of allowed frontend URLs")
print("✅ allow_credentials → needed for cookies/auth")
print("✅ allow_methods → HTTP methods allowed")
print("✅ allow_headers → request headers allowed")
print("")
print("⚠️  NEVER use allow_origins=['*'] with allow_credentials=True!")
print("    This is a security vulnerability.")

In [ ]:
# ============================================================
# Exception Handlers — Global Error Handling
# ============================================================
from pydantic import BaseModel

app4 = FastAPI()

# Custom exception
class NotFoundError(Exception):
    def __init__(self, resource: str, resource_id: int):
        self.resource = resource
        self.resource_id = resource_id

class APIError(BaseModel):
    error: str
    detail: str
    status_code: int

# Register handler for custom exception
@app4.exception_handler(NotFoundError)
async def not_found_handler(request: Request, exc: NotFoundError):
    return JSONResponse(
        status_code=404,
        content=APIError(
            error="not_found",
            detail=f"{exc.resource} with id {exc.resource_id} not found",
            status_code=404
        ).model_dump()
    )

# Catch ALL unhandled exceptions
@app4.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    return JSONResponse(
        status_code=500,
        content=APIError(
            error="internal_error",
            detail="An unexpected error occurred",
            status_code=500
        ).model_dump()
    )

@app4.get("/users/{user_id}")
def get_user(user_id: int):
    if user_id > 100:
        raise NotFoundError("User", user_id)
    return {"id": user_id, "name": "Test User"}

client4 = TestClient(app4, raise_server_exceptions=False)

r = client4.get("/users/1")
print(f"✅ Found: {r.json()}")

r = client4.get("/users/999")
print(f"❌ Not found: {r.json()}")

In [ ]:
# ============================================================
# Lifespan — Startup & Shutdown Events
# ============================================================
from contextlib import asynccontextmanager

# Modern approach (FastAPI 0.93+): lifespan context manager
# Replaces @app.on_event("startup") and @app.on_event("shutdown")

# Fake resources
fake_cache = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # === STARTUP ===
    print("🚀 Starting up...")
    print("  → Connecting to database")
    print("  → Loading ML model")
    print("  → Warming up cache")
    fake_cache["model"] = "loaded"
    
    yield  # ← App runs here
    
    # === SHUTDOWN ===
    print("\n🛑 Shutting down...")
    print("  → Closing database connections")
    print("  → Flushing cache")
    fake_cache.clear()

app5 = FastAPI(lifespan=lifespan)

@app5.get("/status")
def status():
    return {"cache": fake_cache, "status": "running"}

# TestClient triggers lifespan
with TestClient(app5) as client5:
    r = client5.get("/status")
    print(f"\nStatus: {r.json()}")

In [ ]:
# ============================================================
# Putting It All Together — Production App Skeleton
# ============================================================

print("""
# main.py — Production FastAPI App Structure

from contextlib import asynccontextmanager
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
import time, uuid

@asynccontextmanager
async def lifespan(app):
    # Startup: connect DB, load models
    yield
    # Shutdown: close connections

app = FastAPI(title="My API", lifespan=lifespan)

# 1. CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 2. Request logging + timing
@app.middleware("http")
async def log_and_time(request: Request, call_next):
    rid = str(uuid.uuid4())[:8]
    start = time.perf_counter()
    response = await call_next(request)
    ms = (time.perf_counter() - start) * 1000
    response.headers["X-Request-ID"] = rid
    response.headers["X-Process-Time"] = f"{ms:.1f}ms"
    return response

# 3. Global exception handler
@app.exception_handler(Exception)
async def global_handler(request, exc):
    return JSONResponse(status_code=500, content={"error": str(exc)})

# 4. Routes
app.include_router(users_router, prefix="/api/v1")
app.include_router(items_router, prefix="/api/v1")
""")

---

## 🗺️ Middleware & CORS Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  MIDDLEWARE & CORS PATTERNS                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  MIDDLEWARE:  @app.middleware("http")                             ║
║              async def mw(request, call_next):                    ║
║                  response = await call_next(request)              ║
║                  return response                                  ║
║                                                                   ║
║  CORS:       app.add_middleware(CORSMiddleware,                  ║
║                  allow_origins=[...], allow_methods=["*"])        ║
║                                                                   ║
║  EXCEPTION:  @app.exception_handler(MyError)                     ║
║  LIFESPAN:   @asynccontextmanager + yield                        ║
║                                                                   ║
║  ORDER:      CORS → Logging → Timing → Endpoint                 ║
║  SECURITY:   Never origins=['*'] + credentials=True              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is middleware? | Code that runs on every request/response, before and after endpoint |
| 2 | Middleware execution order? | Added last = runs first (stack/LIFO). Response flows back up |
| 3 | What is CORS? | Browser security: blocks cross-origin requests unless server allows it |
| 4 | CORS security mistake? | allow_origins=['*'] + allow_credentials=True → vulnerability |
| 5 | Exception handlers vs middleware? | Exception handlers catch specific errors. Middleware runs on ALL requests |
| 6 | Startup/shutdown events? | Use lifespan context manager (modern). Setup DB, load models, cleanup |
| 7 | How to add request ID? | Middleware: generate UUID, add to response.headers["X-Request-ID"] |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Middleware: before/after EVERY request (timing, auth) │
## │  • CORS: required for frontend←→backend communication   │
## │  • Exception handlers: clean global error responses     │
## │  • Lifespan: replaces on_event (startup/shutdown)       │
## │  • Order matters: last added middleware runs first      │
## │  • Always add X-Request-ID for debugging in production  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 03 — Database Integration with SQLAlchemy**